# Seasonal Accuracy Comparison: RF vs SVM vs XGBoost

This notebook evaluates all three classifiers across **3 phenological seasons**
(Early Summer, Post-Monsoon, Winter) to compare performance stability.

> Uses the same training points, bands, and classifier hyperparameters as the
> individual notebooks in `nb/rf/`, `nb/svm/`, and `nb/xgb/`.

In [ ]:
import sys; sys.path.insert(0, '../nb')
from shared.training_pipeline import init_ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

init_ee()


### Campus Boundary

In [ ]:
import ee
from shared.training_pipeline import get_campus_geometry

campus = get_campus_geometry()


### Cloud Mask & Helpers

In [ ]:
from shared.training_pipeline import mask_s2_clouds, BANDS, SEED

bands = BANDS


### Load Training Points

In [ ]:
from shared.training_pipeline import load_training_points

forest_points, non_forest_points, training_points = load_training_points()

print(f"Forest points:     {forest_points.size().getInfo()}")
print(f"Non-forest points: {non_forest_points.size().getInfo()}")


### Define Seasons & Classifiers

In [ ]:
from shared.training_pipeline import get_classifier_factories

SEASONS = {
    "Early Summer":  ("2025-03-01", "2025-04-30"),
    "Post-Monsoon":  ("2025-09-01", "2025-10-31"),
    "Winter":         ("2025-11-01", "2025-12-31"),
}

print(f"Seasons:     {list(SEASONS.keys())}")
print(f"Classifiers: {list(CLASSIFIERS.keys())}")
print(f"Cloud Cover: {CLOUD_COVER}%")CLOUD_COVER = 1  # best threshold

CLASSIFIERS = get_classifier_factories()

print(f"Seasons:     {list(SEASONS.keys())}")
print(f"Classifiers: {list(CLASSIFIERS.keys())}")
print(f"Cloud Cover: {CLOUD_COVER}%")

### Run Season × Classifier Sweep

In [ ]:
results = []

for season_name, (start_date, end_date) in SEASONS.items():
    print(f"{'='*60}")
    print(f"  Season: {season_name}  ({start_date} → {end_date})")
    print(f"{'='*60}")

    # Build the median composite for this season
    dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterDate(start_date, end_date)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER))
               .map(mask_s2_clouds))

    img_count = dataset.size().getInfo()
    print(f"  Available images: {img_count}")

    if img_count == 0:
        print(f"  ⚠ No images – skipping")
        for clf_name in CLASSIFIERS:
            results.append({
                'season': season_name, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': 0,
            })
        continue

    image = dataset.median().clip(campus)

    # Add NDVI
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    image = image.addBands(ndvi)

    # Sample and split using shared pipeline
    from shared.training_pipeline import sample_and_split
    train_set, test_set = sample_and_split(image, training_points, bands)

    for clf_name, clf_factory in CLASSIFIERS.items():
        try:
            classifier = clf_factory().train(
                features=train_set,
                classProperty='label',
                inputProperties=bands
            )

            validated = test_set.classify(classifier)
            cm = validated.errorMatrix('label', 'classification')
            accuracy = cm.accuracy().getInfo()
            kappa = cm.kappa().getInfo()
            cm_array = cm.getInfo()

            print(f"  {clf_name:25s}  Accuracy: {accuracy:.4f}   Kappa: {kappa:.4f}")
            print(f"  {'':25s}  Confusion Matrix: {cm_array}")

            results.append({
                'season': season_name,
                'classifier': clf_name,
                'accuracy': accuracy,
                'kappa': kappa,
                'confusion_matrix': str(cm_array),
                'image_count': img_count,
            })
        except Exception as e:
            print(f"  {clf_name:25s}  ERROR: {e}")
            results.append({
                'season': season_name, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': img_count,
            })

print("✅ Sweep complete!")

### Results Table

In [ ]:
df = pd.DataFrame(results)
print(df[['season', 'classifier', 'accuracy', 'kappa']].to_string(index=False))

### Accuracy Heatmap (Season × Classifier)

In [ ]:
# Pivot for heatmap
pivot_acc = df.pivot(index='classifier', columns='season', values='accuracy')
pivot_kap = df.pivot(index='classifier', columns='season', values='kappa')

# Reorder columns to match presentation
season_order = ['Early Summer', 'Post-Monsoon', 'Winter']
pivot_acc = pivot_acc[[s for s in season_order if s in pivot_acc.columns]]
pivot_kap = pivot_kap[[s for s in season_order if s in pivot_kap.columns]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy heatmap
sns.heatmap(pivot_acc, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=1, linecolor='white', ax=axes[0],
            vmin=0.5, vmax=1.0)
axes[0].set_title('Model Stability Across Seasons (Accuracy)', fontweight='bold', pad=12)
axes[0].set_ylabel('')
axes[0].set_xlabel('')

# Kappa heatmap
sns.heatmap(pivot_kap, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=1, linecolor='white', ax=axes[1],
            vmin=0.0, vmax=1.0)
axes[1].set_title('Kappa Statistic Across Seasons', fontweight='bold', pad=12)
axes[1].set_ylabel('')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('fe/seasonal_accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/seasonal_accuracy_heatmap.png")

### Bar Chart Comparison

In [ ]:
colors = {
    "Random Forest": "#2ecc71",
    "SVM": "#e74c3c",
    "XGBoost": "#3498db",
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(season_order))
width = 0.25

for i, clf_name in enumerate(CLASSIFIERS):
    subset = df[df['classifier'] == clf_name].set_index('season')
    vals = [subset.loc[s, 'accuracy'] if s in subset.index and pd.notna(subset.loc[s, 'accuracy']) else 0
            for s in season_order]
    ax.bar(x + i * width, vals, width, label=clf_name, color=colors[clf_name])

ax.set_xticks(x + width)
ax.set_xticklabels(season_order)
ax.set_ylabel('Accuracy')
ax.set_title('Classification Accuracy by Season and Classifier', fontweight='bold')
ax.legend()
ax.set_ylim(0.4, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fe/seasonal_accuracy_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/seasonal_accuracy_bars.png")

### Save Results

In [ ]:
df.to_csv('fe/seasonal_accuracy_results.csv', index=False)
print("Saved: fe/seasonal_accuracy_results.csv")

# Summary
print("" + "="*60)
print("  BEST ACCURACY PER SEASON")
print("="*60)
for season in season_order:
    subset = df[(df['season'] == season) & df['accuracy'].notna()]
    if not subset.empty:
        best = subset.loc[subset['accuracy'].idxmax()]
        print(f"  {season:20s}  {best['classifier']:20s}  Accuracy: {best['accuracy']:.4f}")